# Reproducibility Links

Creates a list of links to every notebook (code) and data file (CSV) of the study, grouped by procedure, and saves it as **`reproducibility_links.html`**. The list is meant to be pasted into a Google Doc, which is then shared through a QR code.

The links point to the project's **GitHub repository**, so upload the project first:

1. Create a **public** repository on GitHub and upload the contents of this folder with GitHub Desktop. The `.gitignore` file leaves out the 198 MB boundary shapefile (above GitHub's 100 MB limit) and the download cache.
2. Fill in `GITHUB_USER` and `REPOSITORY` in the settings cell below.
3. Run all cells. The notebook only reads file names; it changes no data.
4. Open `reproducibility_links.html` in a browser, select all (Ctrl + A), copy (Ctrl + C) and paste into a Google Doc (Ctrl + V). Headings, bullets and links are kept.

Run it again whenever files are added or renamed, so the list stays complete.

In [ ]:
from pathlib import Path
from urllib.parse import quote
from collections import Counter
from itertools import groupby
import html
import re

# ============================================================
# SETTINGS - fill in after uploading the project to GitHub
# ============================================================

GITHUB_USER = "gelasanjanpatrick-beep"      
REPOSITORY = "MCS-I-S.Y.26-27-ALLSHS"        # GitHub repository name (no spaces or brackets)
BRANCH = "main"

OUTPUT_FILE = Path("reproducibility_links.html")

STUDY_TITLE = (
    "Development and Evaluation of a Physics-Informed Hybrid SARIMAX-LSTM Model Using "
    "Clear-Sky Index-Informed Learning for Year-Ahead Global Horizontal Irradiance "
    "Forecasting and Spatial Mapping in the Philippines"
)

# File types linked one by one (figures and trained models are linked by folder)
LISTED_TYPES = {".ipynb", ".csv", ".geojson", ".json", ".md", ".txt"}

# Not published (see .gitignore)
EXCLUDED = ["data/boundaries/", "nasa_power_cache_"]

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPOSITORY}"
ZIP_URL = f"{REPO_URL}/archive/refs/heads/{BRANCH}.zip"


def file_url(path):
    return f"{REPO_URL}/blob/{BRANCH}/{quote(path)}"


def folder_url(path):
    return f"{REPO_URL}/tree/{BRANCH}/{quote(path.rstrip('/'))}"


if "YOUR-" in GITHUB_USER + REPOSITORY:
    print("Fill in GITHUB_USER and REPOSITORY first. Until then the links point to placeholders.")
else:
    print("Repository:", REPO_URL)

In [ ]:
# ============================================================
# PROCEDURES - the files of each procedure, in the order of the study
# (an entry ending in "/" covers every file in that folder)
# ============================================================

PROCEDURES = {
    "Overview": [
        "README.md",
        "requirements.txt",
        "Reproducibility_Links.ipynb",
    ],
    "Data acquisition": [
        "Philippine Map Create_Grid.ipynb",
        "Data_Extraction_NASAPOWER.ipynb",
        "Elevation_Extraction.ipynb",
        "data/processed/grid_metadata.csv",
        "data/processed/philippines_grid.geojson",
        "data/raw/nasa_power/nasa_power_monthly.csv",
        "data/raw/elevation/",
    ],
    "Data pre-processing": [
        "Data_Preprocessing_Validation.ipynb",
        "data/processed/nasa_power_validated.csv",
        "data/processed/quality/",
    ],
    "Standalone models and rolling validation": [
        "Standalone_Models.ipynb",
        "outputs/standalone_models/",
    ],
    "Hybrid SARIMAX-LSTM, Ineichen-Perez clear-sky framework, penalized loss and rolling validation": [
        "Hybrid_Models.ipynb",
        "data/processed/clearsky_ineichen_monthly.csv",
        "outputs/hybrid_models/",
    ],
    "Final model testing (2023–2024)": [
        "Final_Testing_Standalone.ipynb",
        "Final_Testing_Hybrid.ipynb",
        "outputs/final_test/",
        "outputs/final_test_hybrid/",
    ],
    "Spatial analysis and mapping": [
        "Final_Test_Visualizations.ipynb",
        "Philippine Map_Grid.ipynb",
        "outputs/figures/final_test_rmse_by_grid.csv",
    ],
    "Comparative and ablation analysis": [
        "Comparative_Ablation_Analysis.ipynb",
        "outputs/significance_tests/",
    ],
    "Final model training and 2026 forecast": [
        "Final_Product_2026.ipynb",
        "data/raw/nasa_power/nasa_power_monthly_2005_2025.csv",
        "data/processed/nasa_power_validated_2005_2025.csv",
        "outputs/final_product_2026/",
    ],
    "Workflow charts": [
        "Workflow_Charts.ipynb",
    ],
}

# Folders linked as a whole
FOLDERS = {
    "outputs/figures/": "All figures: workflow charts, final-test plots and maps, 2026 forecast maps (PNG, SVG, PDF)",
    "outputs/maps/": "Maps of the spatial grid",
    "outputs/standalone_models/models/": "Trained LSTM (.pt) and XGBoost (.json) models, rolling validation",
    "outputs/hybrid_models/models/": "Trained LSTM components of the hybrid models (.pt), rolling validation",
    "outputs/final_test/models/": "Trained LSTM and XGBoost models, final test",
    "outputs/final_test_hybrid/models/": "Trained LSTM components of the hybrid models, final test",
    "outputs/final_product_2026/models/": "Trained LSTM component of the final model, 2025 check and 2026 forecast",
}

# ============================================================
# DESCRIPTIONS
# ============================================================

# Models named in the file names, in the order they are listed
MODELS = {
    "hybrid_csi_pl": "Hybrid SARIMAX-LSTM + CSI + PL (proposed model)",
    "hybrid_csi": "Hybrid SARIMAX-LSTM + CSI",
    "hybrid": "Hybrid SARIMAX-LSTM",
    "sarimax": "SARIMAX",
    "lstm": "LSTM",
    "xgboost": "XGBoost",
    "sarimax_ghi": "SARIMAX (GHI) component",
    "lstm_ghi": "LSTM (GHI) component",
    "sarimax_csi": "SARIMAX (CSI) component",
    "lstm_csi": "LSTM (CSI) component",
    "lstm_csi_pl": "LSTM (CSI + PL) component",
    "climatology": "Climatology baseline",
    "seasonal_naive": "Seasonal naive baseline",
}

# File families: {model} is filled in from the file name
PATTERNS = [
    (r"(component_)?(?P<model>.+)_rolling_predictions\.csv",
     "{model}: forecast vs. observed GHI (kWh/m²/day) for every grid and month, 2015–2022"),
    (r"(component_)?(?P<model>.+)_rolling_metrics_by_grid\.csv",
     "{model}: RMSE, MAE and nRMSE per grid and validation year"),
    (r"(component_)?(?P<model>.+)_rolling_metrics_by_round\.csv",
     "{model}: RMSE, MAE and nRMSE per validation year"),
    (r"(component_)?(?P<model>.+)_final_test_predictions\.csv",
     "{model}: forecast vs. observed GHI (kWh/m²/day) for every grid and month, 2023–2024"),
    (r"(?P<model>.+)_rolling\.csv",
     "{model}: GHI and CSI forecasts for every grid and month, 2014–2022 (2014 is a warm-up year that only sets the first TVWA weights)"),
    (r"(?P<model>.+)_final_test\.csv",
     "{model}: GHI and CSI forecasts for every grid and month, 2023–2024"),
    (r"(?P<model>.+)_tvwa_weights\.csv",
     "{model}: TVWA weights of the SARIMAX and LSTM components per grid and year"),
]

# Files with the same meaning in every folder
BY_NAME = {
    "environment.txt": "Python and library versions used for this run",
    "rolling_metrics_by_grid_all_models.csv": "All models: RMSE, MAE and nRMSE per grid and validation year",
    "rolling_metrics_by_round_all_models.csv": "All models: RMSE, MAE and nRMSE per validation year",
}

# Individual files
BY_PATH = {
    # Overview
    "README.md": "How to run the study: notebook order, data sources and reproducibility notes",
    "requirements.txt": "Python libraries and the exact versions used",
    "Reproducibility_Links.ipynb": "Creates this list of links",

    # Data acquisition
    "Philippine Map Create_Grid.ipynb": "Creates the 1° × 1° grid, keeps the 93 cells that intersect Philippine land and assigns each a representative query point",
    "Data_Extraction_NASAPOWER.ipynb": "Downloads monthly GHI, cloud amount and relative humidity (2005–2024) from NASA POWER for the 93 grids",
    "Elevation_Extraction.ipynb": "Extracts SRTM elevation at each grid's query point through OpenTopoData",
    "data/processed/grid_metadata.csv": "The 93 grid cells: bounds, centre, query point, area and land coverage",
    "data/processed/philippines_grid.geojson": "The 93 grid cells as polygons, for GIS software",
    "data/raw/nasa_power/nasa_power_monthly.csv": "NASA POWER monthly GHI, cloud amount and relative humidity per grid, 2005–2024, as downloaded",
    "data/raw/elevation/grid_elevation.csv": "Elevation (m) of each grid's query point and its source dataset",

    # Data pre-processing
    "Data_Preprocessing_Validation.ipynb": "Checks structure, missing values, physical validity, outliers and other quality issues; adds extraterrestrial irradiance (H0) and the clearness index (Kt)",
    "data/processed/nasa_power_validated.csv": "Validated dataset, 2005–2024, with H0, Kt and quality flags (input of the models)",
    "data/processed/quality/validation_summary.csv": "Result of each of the 25 data-quality checks",
    "data/processed/quality/flagged_records.csv": "Values flagged as outliers (|modified z| > 3.5), with details",

    # Standalone models and rolling validation
    "Standalone_Models.ipynb": "SARIMAX, LSTM and XGBoost, with climatology and seasonal-naive baselines; rolling validation 2015–2022",
    "outputs/standalone_models/rolling_rounds.csv": "The 8 rolling-validation rounds: training period and validation year",
    "outputs/standalone_models/rolling_metrics/rolling_metrics_summary.csv": "Summary of every model: pooled RMSE, MAE, nRMSE and skill vs. climatology, 2015–2022",
    "outputs/standalone_models/models/sarimax_rolling_diagnostics.csv": "SARIMAX order, AIC, convergence and Ljung-Box p-value per grid and round",
    "outputs/standalone_models/models/lstm_training_history.csv": "LSTM training and early-stopping loss per epoch and round",
    "outputs/standalone_models/models/xgboost_feature_importance_last_round.csv": "XGBoost feature importance (gain), last round",

    # Hybrid models
    "Hybrid_Models.ipynb": "Ineichen-Perez clear-sky GHI, the hybrid SARIMAX-LSTM (TVWA), + CSI and + penalized loss; rolling validation 2015–2022",
    "data/processed/clearsky_ineichen_monthly.csv": "Ineichen-Perez clear-sky GHI (kWh/m²/day) per grid and month, 2005–2026",
    "outputs/hybrid_models/rolling_metrics/rolling_metrics_summary.csv": "Summary of every model: pooled RMSE, MAE, nRMSE, clear-sky violations and skill vs. climatology, 2015–2022",
    "outputs/hybrid_models/models/sarimax_component_orders.csv": "SARIMAX component orders per grid (GHI and CSI), chosen by ADF test and AIC",
    "outputs/hybrid_models/models/sarimax_component_diagnostics_rolling.csv": "SARIMAX component AIC, convergence and Ljung-Box p-value per grid and year",

    # Final model testing
    "Final_Testing_Standalone.ipynb": "Final test of SARIMAX, LSTM, XGBoost and the two baselines on 2023–2024",
    "Final_Testing_Hybrid.ipynb": "Final test of the three hybrid models and their components on 2023–2024",
    "outputs/final_test/final_test_metrics.csv": "Summary of every model: pooled RMSE, MAE, nRMSE, R² and skill vs. climatology, 2023–2024",
    "outputs/final_test/final_test_metrics_by_grid.csv": "RMSE, MAE, nRMSE and R² per grid and model, 2023–2024",
    "outputs/final_test/models/sarimax_final_test_diagnostics.csv": "SARIMAX order, AIC, convergence and Ljung-Box p-value per grid, 2023 and 2024",
    "outputs/final_test/models/lstm_final_test_training_history.csv": "LSTM training and early-stopping loss per epoch, 2023 and 2024",
    "outputs/final_test_hybrid/final_test_metrics.csv": "Summary of every model: pooled RMSE, MAE, nRMSE, R², clear-sky violations and skill vs. climatology, 2023–2024",
    "outputs/final_test_hybrid/final_test_metrics_by_grid.csv": "RMSE, MAE, nRMSE, R² and clear-sky violations per grid and model, 2023–2024",
    "outputs/final_test_hybrid/models/sarimax_component_diagnostics_final_test.csv": "SARIMAX component AIC, convergence and Ljung-Box p-value per grid, 2023 and 2024",

    # Spatial analysis and mapping
    "Final_Test_Visualizations.ipynb": "R² scatter plot and RMSE maps of the final test",
    "Philippine Map_Grid.ipynb": "Maps of the spatial grid, with and without grid numbers",
    "outputs/figures/final_test_rmse_by_grid.csv": "Final-test RMSE per grid of the six models (data of the RMSE maps)",

    # Comparative and ablation analysis
    "Comparative_Ablation_Analysis.ipynb": "Diebold-Mariano tests with HLN correction for the comparative and ablation analyses (2015–2024), and RMSE difference maps",
    "outputs/significance_tests/model_accuracy_full_record.csv": "RMSE, MAE and nRMSE of every model over the full record, 2015–2024",
    "outputs/significance_tests/comparative_dm_hln.csv": "Comparative analysis: proposed model vs. SARIMAX, LSTM, XGBoost, climatology and seasonal naive (DM-HLN, Holm-adjusted)",
    "outputs/significance_tests/ablation_dm_hln.csv": "Ablation analysis: the three hybrid variants compared (DM-HLN, Holm-adjusted)",
    "outputs/significance_tests/per_grid_dm_hln.csv": "DM-HLN test per grid for each comparison (Benjamini-Hochberg FDR-adjusted)",
    "outputs/significance_tests/loss_differentials_monthly.csv": "Monthly loss differentials used in the tests, 2015–2024",

    # Final model training and 2026 forecast
    "Final_Product_2026.ipynb": "Checks the proposed model on 2025, retrains it on 2005–2025 and forecasts 2026, with maps",
    "data/raw/nasa_power/nasa_power_monthly_2005_2025.csv": "NASA POWER monthly GHI, cloud amount and relative humidity per grid, 2005–2025, as downloaded for the final model",
    "data/processed/nasa_power_validated_2005_2025.csv": "Validated dataset, 2005–2025, with H0, Kt, elevation and month terms (input of the final model)",
    "outputs/final_product_2026/forecast_2026_monthly.csv": "2026 forecast: monthly GHI per grid, with clear-sky GHI, CSI and the two component forecasts",
    "outputs/final_product_2026/forecast_2026_annual.csv": "2026 forecast: annual mean daily GHI and annual total GHI per grid, compared with the 2005–2025 average",
    "outputs/final_product_2026/check_2025_predictions.csv": "2025 check: forecast vs. observed GHI per grid and month",
    "outputs/final_product_2026/check_2025_metrics.csv": "2025 check: RMSE, MAE, nRMSE, R² and skill of the proposed model, its two components and climatology",
    "outputs/final_product_2026/nasa_power_revisions_vs_development.csv": "Comparison of the 2005–2024 values of the two NASA POWER downloads (no value was revised)",
    "outputs/final_product_2026/nasa_power_request_header.json": "NASA POWER API version, sources and period of the 2005–2025 download",
    "outputs/final_product_2026/models/tvwa_weights.csv": "TVWA weights per grid, 2025 and 2026",
    "outputs/final_product_2026/models/sarimax_csi_diagnostics.csv": "SARIMAX (CSI) order, AIC, convergence and Ljung-Box p-value per grid, 2025 and 2026",

    # Workflow charts
    "Workflow_Charts.ipynb": "Workflow charts of the procedures",
}

# Listed first (in this order) or last within their folder
FIRST = [
    "validation_summary.csv",
    "rolling_rounds.csv", "rolling_metrics_summary.csv",
    "final_test_metrics.csv", "final_test_metrics_by_grid.csv",
    "model_accuracy_full_record.csv", "comparative_dm_hln.csv", "ablation_dm_hln.csv", "per_grid_dm_hln.csv",
    "forecast_2026_monthly.csv", "forecast_2026_annual.csv", "check_2025_predictions.csv", "check_2025_metrics.csv",
]
LAST = ["environment.txt"]

In [ ]:
# ============================================================
# FIND AND DESCRIBE THE FILES
# ============================================================

def is_listed(path):
    folders = Path(path).parts[:-1]
    if any(part.startswith(".") or part in ("venv", "__pycache__") for part in folders):
        return False                                   # Git, checkpoint and environment folders
    if any(text in path for text in EXCLUDED):
        return False                                   # not published
    if "/models/" in path and not path.endswith(".csv"):
        return False                                   # trained models are linked by folder
    return Path(path).suffix in LISTED_TYPES


def entry_of(path):
    """Procedure of a file and the position of its entry in PROCEDURES"""
    for procedure, entries in PROCEDURES.items():
        for position, entry in enumerate(entries):
            if path == entry or (entry.endswith("/") and path.startswith(entry)):
                return procedure, position, entry
    return "Other files", 0, ""


def model_of(path):
    for pattern, _ in PATTERNS:
        match = re.fullmatch(pattern, Path(path).name)
        if match:
            return match.group("model")
    return None


def describe(path):
    name = Path(path).name
    if path in BY_PATH:
        return BY_PATH[path]
    if name in BY_NAME:
        return BY_NAME[name]
    for pattern, text in PATTERNS:
        match = re.fullmatch(pattern, name)
        if match:
            model = match.group("model")
            return text.format(model=MODELS.get(model, model))
    return ""


SUBFOLDER_ORDER = ["", "rolling_predictions", "predictions", "components", "rolling_metrics", "models"]


def sort_key(path):
    """Order of the study: entry in PROCEDURES, sub-folder, then model"""
    _, position, entry = entry_of(path)
    inside = path[len(entry):] if entry.endswith("/") else ""
    subfolder = inside.split("/")[0] if "/" in inside else ""
    subfolder_rank = SUBFOLDER_ORDER.index(subfolder) if subfolder in SUBFOLDER_ORDER else len(SUBFOLDER_ORDER)
    name = Path(path).name
    if name in FIRST:
        name_rank = FIRST.index(name) - len(FIRST)
    elif name in LAST:
        name_rank = 1000
    else:
        model = model_of(path)
        name_rank = list(MODELS).index(model) + 1 if model in MODELS else 0
    return position, subfolder_rank, name_rank, path


files = sorted(
    (p.as_posix() for p in Path(".").rglob("*") if p.is_file() and is_listed(p.as_posix())),
    key=sort_key,
)

types = Counter(Path(p).suffix for p in files)
print(f"Files to link: {len(files)} (" + ", ".join(f"{n} {t}" for t, n in sorted(types.items())) + ")")

for procedure in [*PROCEDURES, "Other files"]:
    n = sum(entry_of(p)[0] == procedure for p in files)
    if n:
        print(f"  {n:3d}  {procedure}")

for path in files:
    if entry_of(path)[0] == "Other files":
        print("Not in any procedure (listed under 'Other files'):", path)
    if not describe(path):
        print("No description:", path)

In [ ]:
# ============================================================
# WRITE THE HTML PAGE
# ============================================================

def link(url, text):
    return f'<a href="{html.escape(url)}">{html.escape(text)}</a>'


def item(url, text, description):
    return f"<li>{link(url, text)} – {html.escape(description)}</li>" if description else f"<li>{link(url, text)}</li>"


def folder_label(path):
    folder = Path(path).parent.as_posix()
    if folder == ".":
        return "Code" if path.endswith(".ipynb") else ""
    return folder + "/"


parts = [
    "<h1>Code and Data for Reproducibility</h1>",
    f"<p><i>{html.escape(STUDY_TITLE)}</i></p>",
    "<p>Each link opens a file on GitHub, where it can be viewed or downloaded. "
    "Notebooks (.ipynb) show the code with its outputs, and small CSV files show as a table. "
    "README.md gives the order in which to run the notebooks.</p>",
]

for procedure in [*PROCEDURES, "Other files"]:
    paths = [p for p in files if entry_of(p)[0] == procedure]
    if not paths:
        continue
    parts.append(f"<h2>{html.escape(procedure)}</h2>")

    if procedure == "Overview":
        rows = [item(REPO_URL, "Whole repository", "every file of the study"),
                item(ZIP_URL, "Download everything (ZIP)", "the whole repository as one file")]
        rows += [item(file_url(p), Path(p).name, describe(p)) for p in paths]
        parts += ["<ul>", *rows, "</ul>"]
        continue

    for label, block in groupby(paths, key=folder_label):
        if label == "Code":
            parts.append("<p><b>Code</b></p>")
        elif label:
            parts.append(f"<p><b>{link(folder_url(label), label)}</b></p>")
        parts += ["<ul>", *(item(file_url(p), Path(p).name, describe(p)) for p in block), "</ul>"]

folders = [(folder, text) for folder, text in FOLDERS.items() if Path(folder).is_dir()]
if folders:
    parts.append("<h2>Figures and trained models</h2>")
    parts += ["<ul>", *(item(folder_url(folder), folder, text) for folder, text in folders), "</ul>"]

parts.append(f"<p>All files are in the {html.escape(BRANCH)} branch of {link(REPO_URL, REPO_URL)}.</p>")

page = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Code and Data for Reproducibility</title>
<style>
  body { font-family: Arial, Helvetica, sans-serif; font-size: 11pt; line-height: 1.45;
         color: #202124; background: #ffffff; max-width: 52rem; margin: 0 auto; padding: 1.5rem 1rem; }
  h1 { font-size: 20pt; font-weight: normal; margin: 0 0 0.4em; }
  h2 { font-size: 16pt; font-weight: normal; margin: 1.6em 0 0.4em; }
  p { margin: 0.5em 0; }
  ul { margin: 0.2em 0 0.6em; padding-left: 1.6em; }
  a { color: #1155cc; overflow-wrap: anywhere; }
</style>
</head>
<body>
""" + "\n".join(parts) + """
</body>
</html>
"""

OUTPUT_FILE.write_text(page, encoding="utf-8")

n_links = page.count("<a href=")
print(f"Saved {OUTPUT_FILE} with {n_links} links")
print("Example:", file_url("README.md"))

## Next steps

1. **Check a few links**: open `reproducibility_links.html` in a browser and click some of them. Each should open the file on GitHub. If you get "404 - page not found", check `GITHUB_USER`, `REPOSITORY` and `BRANCH`, and make sure the repository is public.
2. **Google Doc**: select all on the page (Ctrl + A), copy (Ctrl + C), and paste into a new Google Doc (Ctrl + V).
3. **Share the Doc**: click Share, then under General access choose "Anyone with the link" with the Viewer role, and copy the link.
4. **QR code**: make a QR code of the Doc's link, for example in Chrome (open the Doc, then Share → Create QR Code). Scan it with a phone to test it before putting it in the paper.

Because the QR code points to the Doc, you can still edit the Doc later without changing the QR code.